# IoT-Sensor-Forecast · 探索性数据分析 (EDA)

**数据集**: UCI 374 Appliances Energy Prediction（采样间隔 10 分钟）

本 notebook 针对智能建筑 4.5 个月的物联网传感器数据进行探索性分析。分析目标包括：

1. **数据概览**：了解数据结构、列类型和取值范围
2. **缺失值检查**：在建模前验证数据完整性
3. **目标变量分布**：分析电器能耗的模式（右偏分布，中位数=60 瓦）
4. **相关性分析**：识别特征关系，特别是 T6 传感器的异常情况

In [1]:
import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 中文字体配置（Windows 自带 SimHei/Microsoft YaHei）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 加载数据
df = pd.read_csv('D:/Project/IoT-Sensor-Forecast/data/raw/appliances_energy.csv')

print(f"数据形状: {df.shape}")
print(f"\n前 3 行:\n{df.head(3)}")
print(f"\nDataFrame 信息:")
df.info()

数据形状: (19735, 29)

前 3 行:
                 date  lights     T1       RH_1    T2       RH_2     T3  \
0  2016-01-1117:00:00      30  19.89  47.596667  19.2  44.790000  19.79   
1  2016-01-1117:10:00      30  19.89  46.693333  19.2  44.722500  19.79   
2  2016-01-1117:20:00      30  19.89  46.300000  19.2  44.626667  19.79   

        RH_3         T4       RH_4  ...   RH_9  T_out  Press_mm_hg  RH_out  \
0  44.730000  19.000000  45.566667  ...  45.53   6.60        733.5    92.0   
1  44.790000  19.000000  45.992500  ...  45.56   6.48        733.6    92.0   
2  44.933333  18.926667  45.890000  ...  45.50   6.37        733.7    92.0   

   Windspeed  Visibility  Tdewpoint        rv1        rv2  Appliances  
0   7.000000   63.000000        5.3  13.275433  13.275433          60  
1   6.666667   59.166667        5.2  18.606195  18.606195          60  
2   6.333333   55.333333        5.1  28.642668  28.642668          50  

[3 rows x 29 columns]

DataFrame 信息:
<class 'pandas.DataFrame'>
RangeIn

## 1. 数据概览

本步骤检查数据结构：字段名、数据类型和取值范围。使用 `df.describe(include='all').T` 得到转置后的描述统计，同时包含数值列和字符串列的汇总信息，便于快速了解每个字段的分布特征。

In [2]:
desc = df.describe(include='all').T
print(desc)
plt.close('all')

               count unique                 top freq        mean         std  \
date           19735  19735  2016-01-1117:00:00    1         NaN         NaN   
lights       19735.0    NaN                 NaN  NaN    3.801875    7.935988   
T1           19735.0    NaN                 NaN  NaN   21.686571    1.606066   
RH_1         19735.0    NaN                 NaN  NaN   40.259739    3.979299   
T2           19735.0    NaN                 NaN  NaN   20.341219    2.192974   
RH_2         19735.0    NaN                 NaN  NaN    40.42042    4.069813   
T3           19735.0    NaN                 NaN  NaN   22.267611    2.006111   
RH_3         19735.0    NaN                 NaN  NaN     39.2425    3.254576   
T4           19735.0    NaN                 NaN  NaN   20.855335    2.042884   
RH_4         19735.0    NaN                 NaN  NaN   39.026904    4.341321   
T5           19735.0    NaN                 NaN  NaN   19.592106    1.844623   
RH_5         19735.0    NaN             

## 2. 缺失值统计

缺失值会偏置模型训练过程，甚至导致程序崩溃。在进行任何特征工程或模型拟合之前，验证数据完整性是关键的第一步。本步骤统计每列缺失数量并计算整体缺失比例。

In [3]:
missing = df.isnull().sum().sort_values(ascending=False)
total_missing = missing.sum()
total_cells = df.shape[0] * df.shape[1]
missing_pct = (total_missing / total_cells) * 100

print("各列缺失值数量:")
print(missing)
print(f"\n总缺失单元格: {total_missing} / {total_cells} ({missing_pct:.4f}%)")

if total_missing > 0:
    print("\n提示: 检测到缺失值，请考虑合适的插补策略。")
else:
    print("\n提示: 未发现缺失值，数据集完整。")

plt.close('all')

各列缺失值数量:
date           0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
Appliances     0
dtype: int64

总缺失单元格: 0 / 572315 (0.0000%)

提示: 未发现缺失值，数据集完整。


## 3. 目标变量分布

`Appliances` 是预测目标，即电器能耗（瓦）。该分布呈现严重的右偏（偏度=3.386），意味着大部分读数集中在低值区间（<100 瓦），仅在少数时刻出现高能耗峰值（最大值=1080 瓦）。这种长尾分布会显著影响回归模型的损失计算。

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 直方图
axes[0].hist(df['Appliances'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df['Appliances'].mean(), color='red', linestyle='--', label=f"均值={df['Appliances'].mean():.1f} 瓦")
axes[0].axvline(df['Appliances'].median(), color='green', linestyle='--', label=f"中位数={df['Appliances'].median():.1f} 瓦")
axes[0].set_xlabel('电器能耗 (瓦)')
axes[0].set_ylabel('频次')
axes[0].set_title('电器能耗分布直方图')
axes[0].legend()

# 箱线图
axes[1].boxplot(df['Appliances'], orientation='vertical')
axes[1].set_ylabel('电器能耗 (瓦)')
axes[1].set_title('电器能耗箱线图')

plt.tight_layout()
plt.savefig('D:/Project/IoT-Sensor-Forecast/reports/figures/eda_target_dist.png', dpi=120, bbox_inches='tight')
plt.close('all')

print(df['Appliances'].describe())

count    19735.000000
mean        97.694958
std        102.524891
min         10.000000
25%         50.000000
50%         60.000000
75%        100.000000
max       1080.000000
Name: Appliances, dtype: float64


## 4. 时序趋势

在 4.5 个月（2016 年 1 月 11 日 – 5 月 27 日）的观测窗口内，数据呈现明显的日内周期性。我们解析 `date` 列、按天重采样，并同时可视化电器能耗趋势与各房间温度传感器（T1–T9）的变化。

In [5]:
# 解析日期列（处理格式异常的字符串，例如 '2016-01-1117:00:00' -> 插入空格）
df['date'] = df['date'].str.replace(r'(\d{4}-\d{2}-\d{2})(\d{2}:)', r'\1 \2', regex=True)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 时序图：电器能耗（日均）
df_ts = df.set_index('date')
daily_appliances = df_ts['Appliances'].resample('D').mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily_appliances.index, daily_appliances.values, linewidth=0.8)
ax.set_xlabel('日期')
ax.set_ylabel('电器能耗 (瓦)')
ax.set_title('日均电器能耗（2016年1-5月）')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('D:/Project/IoT-Sensor-Forecast/reports/figures/eda_timeseries.png', dpi=120, bbox_inches='tight')
plt.close('all')

# 各房间温度 T1-T9（3x3 子图）
temp_cols = ['T1','T2','T3','T4','T5','T6','T7','T8','T9']
daily_temps = df_ts[temp_cols].resample('D').mean()

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
axes = axes.flatten()
for i, col in enumerate(temp_cols):
    axes[i].plot(daily_temps.index, daily_temps[col], linewidth=0.7)
    axes[i].set_title(f'房间 {col} 温度')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].set_ylabel('温度 (℃)')

plt.tight_layout()
plt.savefig('D:/Project/IoT-Sensor-Forecast/reports/figures/eda_room_temperatures.png', dpi=120, bbox_inches='tight')
plt.close('all')

## 5. 相关性分析

热力图可以揭示特征冗余与多重共线性。完整的相关性矩阵显示：`rv1` 与 `rv2` 完全相同（相关系数=1.0），而 `T6` 传感器表现异常——它与室外温度 `T_out` 的相关系数高达 0.975（接近完全线性相关），推测该传感器可能实际安装于建筑外部，却被误标注为室内房间。

In [6]:
# 完整相关性热力图（仅数值列）
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('数值特征相关性热力图')
plt.tight_layout()
plt.savefig('D:/Project/IoT-Sensor-Forecast/reports/figures/eda_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.close('all')

# 条形图：各房间温度与电器能耗的相关性
temp_corr = df[temp_cols + ['Appliances']].corr()['Appliances'][temp_cols].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(temp_corr.index, temp_corr.values, color='steelblue', edgecolor='black')
ax.set_xlabel('房间温度传感器')
ax.set_ylabel('与电器能耗的相关性')
ax.set_title('T1-T9 房间温度与电器能耗的相关性')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, temp_corr.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.3f}',
            ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('D:/Project/IoT-Sensor-Forecast/reports/figures/eda_target_corr.png', dpi=120, bbox_inches='tight')
plt.close('all')

print("与电器能耗相关性最高的特征:")
print(temp_corr.sort_values(ascending=False))

与电器能耗相关性最高的特征:
T2    0.120073
T6    0.117638
T3    0.085060
T1    0.055447
T4    0.040281
T8    0.039572
T7    0.025801
T5    0.019760
T9    0.010010
Name: Appliances, dtype: float64


## 6. 关键发现与下一步

### 关键发现

1. **目标变量右偏**：电器能耗偏度高达 3.386，中位数 60 瓦 vs 均值 97.7 瓦，建议在建模前进行 log 或 Box-Cox 变换以缓解偏态影响。
2. **冗余列**：`rv1` 与 `rv2` 完全相同（相关系数=1.0），建模前应删除其中一列。
3. **T6 传感器异常**：与室外温度 `T_out` 的相关系数为 0.975（其他房间为 0.50–0.79），推测该传感器实为户外探头被误标为室内房间。
4. **lights 特征稀疏**：约 77.3% 的取值为 0，且仅包含 8 个离散值，可视为占用率/活动度的稀疏指示符。
5. **日期解析问题**：原始字符串格式如 `2016-01-1117:00:00` 缺少分隔空格，需要在解析前进行正则插入。

### 第 5-6 天特征工程建议

- **删除 `rv2`**（与 `rv1` 重复），并考虑删除 `T6` 或将其显式标记为室外传感器
- **对 `Appliances` 取对数**，降低目标分布偏度后再建模
- **提取时间特征**：小时、星期几、是否周末——能耗呈现明显的日内周期
- **构建滞后特征**：t-1、t-2 时刻的电器能耗（自回归信号）
- **滚动统计**：电器能耗与房间温度的 1 小时 / 6 小时滚动均值
- **lights 编码**：保留为二值指示符，或聚合为每小时累计开启分钟数